https://www.kaggle.com/datasets/yasserhessein/dataset-alzheimer

# IMPORTS

In [ ]:
!rm -r "/content/dataset-alzheimer.zip" #if already exists delete
!rm -r "/content/dataset"

In [ ]:
!pip install tensorflow
!pip install albumentations
!pip install --upgrade tensorflow

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv2D,Dense,Dropout,BatchNormalization,Flatten,MaxPool2D,MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau,ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ResNet50,DenseNet121,EfficientNetB2,VGG19,Xception,VGG16
from tensorflow.keras import layers
from tensorflow.keras import Model
import cv2

# DATA Download

In [ ]:
!rm -r "/content/dataset-alzheimer.zip" #if already exists delete

In [ ]:
!pip install -q kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json     #giving permission
!kaggle datasets download yasserhessein/dataset-alzheimer
!unzip "/content/dataset-alzheimer.zip" -d "/content/dataset/"  # UNZIP
!rm -r "/content/dataset-alzheimer.zip"

# DATA MANAGEMENT

## Changing data

# Model

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sn
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Flatten, Dropout, Activation, Conv2D, MaxPooling2D, LeakyReLU
from tensorflow.keras.callbacks import TensorBoard
import pickle
from PIL import Image
import time
import os
from PIL import Image
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Device:', tpu.master())
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
except:
    strategy = tf.distribute.get_strategy()

In [ ]:
IMAGE_SIZE = [176,208]               # input image dimensions                                              (176,208)
BATCH_SIZE = 5000
test_split_percent = .1       # % of total data for testing                                         .1
validation_split_percent = .2 # % of total data for validation                                      .2
zoom = [.99,1.01]             # zoom range (for a fixed zoom put the same value in both parameters)[.99,1.01]
bright_range = [.8,1.2]       # brightness range                                                    [.8,1.2]
layers_unlocked = True        # unlock the imported pre-training layers?                            False
lr = 0.01                   # learning rate for optimizer                                         0.0001
batch = 20                    # batch size for model fitting                                        20
EPOCHS = 50                     # number of epochs to run                                             50

## train & test

In [ ]:
  # directoryKaggle
  !mkdir -p ~/.kaggle

  from google.colab import files
  files.upload()
  !mv kaggle.json ~/.kaggle/
  # permissions imp
  !chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d yasserhessein/dataset-alzheimer

In [ ]:
import zipfile

with zipfile.ZipFile("dataset-alzheimer.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset-alzheimer")


In [ ]:
train_dr = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255,fill_mode='constant',cval=0,
                                                           brightness_range=bright_range,zoom_range=zoom,
                                                           data_format='channels_last',zca_whitening=False)

train_data_gen = train_dr.flow_from_directory(directory="/content/dataset-alzheimer/Alzheimer_s Dataset/train",target_size=IMAGE_SIZE,batch_size=BATCH_SIZE)

# Change to zoom = [1,1] to use normal test data
test_dr = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255,fill_mode='constant',cval=0,zoom_range=[1,1],
                                                          data_format='channels_last')
test_data_gen = test_dr.flow_from_directory(directory="/content/dataset-alzheimer/Alzheimer_s Dataset/test",target_size=IMAGE_SIZE,batch_size=BATCH_SIZE,
                                           shuffle = False) # test data should not be shuffle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_data,train_labels =  train_data_gen.__next__()
test_data,test_labels = test_data_gen.__next__()

In [ ]:
total_data = np.concatenate((train_data,test_data))
total_labels = np.concatenate((train_labels,test_labels))
print(total_data.shape)
print(total_labels.shape)

In [ ]:
print(total_labels)

In [ ]:
initial_split = test_split_percent+validation_split_percent
test_val_split = test_split_percent/initial_split

# split into training and (test + validation)
train_data, test_val_data, train_labels, test_val_labels = train_test_split(total_data,total_labels,
                                                                            test_size=initial_split)

# split (test + validation) into test and validation sets
test_data, val_data, test_labels, val_labels = train_test_split(test_val_data,test_val_labels,
                                                                test_size=test_val_split)

print('train: ',train_data.shape)
print('validation',val_data.shape)
print('test',test_data.shape)

In [ ]:
print(train_data.shape)
print(train_labels.shape)
print(val_data.shape)
print(val_labels.shape)
print(test_data.shape)
print(test_labels.shape)

In [ ]:
plt.subplot(221)
plt.imshow(train_data[1,:,:,:])
plt.subplot(222)
plt.imshow(train_data[2,:,:,:])
plt.subplot(223)
plt.imshow(val_data[3,:,:,:])
plt.subplot(224)
plt.imshow(val_data[4,:,:,:])
plt.show()
plt.subplot(221)
plt.imshow(test_data[5,:,:,:])
plt.subplot(222)
plt.imshow(test_data[154,:,:,:])

In [ ]:
print(np.amax(train_data))
print(np.amin(train_data))
print(np.amax(val_data))
print(np.amin(val_data))

In [ ]:
plt.subplot(141)
plt.imshow(train_data[3,:,:,0])
plt.subplot(142)
plt.imshow(train_data[3,:,:,1])
plt.subplot(143)
plt.imshow(train_data[3,:,:,2])
plt.subplot(144)
plt.imshow(train_data[3,:,:,:])

## filters

In [ ]:
def conv_block(filters):
    block = tf.keras.Sequential([
        tf.keras.layers.SeparableConv2D(filters, 3, activation='relu', padding='same'),
        tf.keras.layers.SeparableConv2D(filters, 3, activation='relu', padding='same'),
        #tf.keras.layers.SeparableConv2D(filters, 3, activation='relu', strides=2, padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D()
    ])

    return block

In [ ]:
def dense_block(units, dropout_rate):
    block = tf.keras.Sequential([
        tf.keras.layers.Dense(units, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(dropout_rate)
    ])

    return block

## model 1

In [ ]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(*IMAGE_SIZE, 3)),

        tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        #tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPool2D(),

        conv_block(32),
        conv_block(64),

        conv_block(128),
        tf.keras.layers.Dropout(0.2),

        conv_block(256),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Flatten(),
        dense_block(512, 0.7),
        dense_block(128, 0.5),
        dense_block(64, 0.3),

        tf.keras.layers.Dense(4, activation='softmax')
    ])

    return model

In [ ]:
with strategy.scope():
    model = build_model()

    METRICS = [tf.keras.metrics.BinaryAccuracy(name='accuracy'),tf.keras.metrics.Precision(name='precision'),
               tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.AUC(name='auc')]


    model.compile(
        optimizer='adam',
        loss=tf.losses.CategoricalCrossentropy(),
        metrics=METRICS
    )


In [ ]:
def exponential_decay(lr0, s):
    def exponential_decay_fn(epoch):
        return lr0 * 0.1 **(epoch / s)
    return exponential_decay_fn

exponential_decay_fn = exponential_decay(0.01, 20)

lr_scheduler = tf.keras.callbacks.LearningRateScheduler(exponential_decay_fn)

# checkpoint_cb = tf.keras.callbacks.ModelCheckpoint("/content/alzheimer_model.keras", save_best_only=True)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "/content/alzheimer_model.keras",  # Path to save the model
    save_best_only=True,               # Save only the best model (based on a metric)
    save_weights_only=False,           # Save both architecture + weights (set to True for only weights)
    monitor='val_loss',                # Monitor validation loss or any other metric
    mode='min',                        # Mode for monitoring (e.g., 'min' to minimize val_loss)
    verbose=1                          # Show progress in the output
)

early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

In [ ]:
model.summary()

### train

In [ ]:
model_history = model.fit(train_data,train_labels,validation_data=(val_data,val_labels),
                             epochs=EPOCHS,batch_size=batch, shuffle=True)

In [ ]:
model.save('/content/full_model.keras')

In [ ]:
model = tf.keras.models.load_model('/content/full_model.keras')


### Test


In [ ]:
for layer in model.layers:
    print(layer.name)


In [ ]:
for layer in model.get_layer("sequential_3").layers:
    print(layer.name)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Select an image from the test data
image_index = 0  # Change this to view other images
image = test_data[image_index]
true_label = test_labels[image_index]

# Preprocess the image
input_image = np.expand_dims(image, axis=0)
input_image = tf.convert_to_tensor(input_image)

# Compute gradients of the predicted class with respect to the input image
with tf.GradientTape() as tape:
    tape.watch(input_image)
    predictions = model(input_image)
    predicted_label = tf.argmax(predictions[0])
    predicted_score = predictions[0][predicted_label]

# Get the gradients of the predicted class score with respect to the input image
grads = tape.gradient(predicted_score, input_image)

# Take the maximum gradient across the RGB channels
saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0]

# Normalize the saliency map
saliency = (saliency - tf.reduce_min(saliency)) / (tf.reduce_max(saliency) - tf.reduce_min(saliency))

# Plot the saliency map
plt.figure(figsize=(12, 6))

# Original image
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title(f"True Label: {np.argmax(true_label)}, Predicted Label: {predicted_label}")
plt.axis('off')

# Saliency map
plt.subplot(1, 2, 2)
plt.imshow(saliency, cmap="hot")
plt.title("Saliency Map")
plt.axis('off')

plt.show()


In [ ]:
# Separate test data into positive and negative cases
positive_indices = np.where(np.argmax(test_labels, axis=1) == 0)[0]  # Adjust based on your label encoding
negative_indices = np.where(np.argmax(test_labels, axis=1) == 1)[0]

positive_images = test_data[positive_indices]
negative_images = test_data[negative_indices]


In [ ]:
def generate_saliency_map(model, image):
    input_image = np.expand_dims(image, axis=0)
    input_image = tf.convert_to_tensor(input_image)

    # Compute gradients
    with tf.GradientTape() as tape:
        tape.watch(input_image)
        predictions = model(input_image)
        predicted_label = tf.argmax(predictions[0])
        predicted_score = predictions[0][predicted_label]

    grads = tape.gradient(predicted_score, input_image)
    saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0]
    saliency = (saliency - tf.reduce_min(saliency)) / (tf.reduce_max(saliency) - tf.reduce_min(saliency))

    return saliency
n=500
# Generate saliency maps for a subset of images
positive_saliency_maps = [generate_saliency_map(model, img) for img in positive_images[:n]]
negative_saliency_maps = [generate_saliency_map(model, img) for img in negative_images[:n]]


In [ ]:
def plot_saliency(image, saliency, title):
    plt.figure(figsize=(10, 5))

    # Original Image
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis("off")

    # Saliency Map
    plt.subplot(1, 2, 2)
    plt.imshow(image)
    plt.imshow(saliency, cmap='hot', alpha=0.5)
    plt.title(title)
    plt.axis("off")

    plt.show()

# Plot examples
for i in range(10):  # Adjust the range to visualize more examples
    plot_saliency(positive_images[i], positive_saliency_maps[i], "Positive Case")
    plot_saliency(negative_images[i], negative_saliency_maps[i], "Negative Case")


In [ ]:
# Compute average saliency maps
avg_positive_saliency = np.mean(positive_saliency_maps, axis=0)
avg_negative_saliency = np.mean(negative_saliency_maps, axis=0)

# Plot average saliency maps
plot_saliency(avg_positive_saliency, avg_positive_saliency, "Average Positive Case")
plot_saliency(avg_negative_saliency, avg_negative_saliency, "Average Negative Case")


In [ ]:
# Evaluate the model on the test dataset
test_loss, test_accuracy, test_precision, test_recall, test_auc = model.evaluate(test_data, test_labels, batch_size=batch)

# Print the results
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test AUC: {test_auc:.4f}")


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def compute_gradcam(model, image, class_idx):
    # Create a model that outputs the last conv layer and the predictions
    last_conv_layer_name = "sequential_3"  # Update with the name of your last conv layer
    grad_model = tf.keras.models.Model(
    [model.inputs],
    [model.get_layer(last_conv_layer_name).output,
    model.get_layer("sequential_3").output])

    # Add batch dimension to the image
    image = np.expand_dims(image, axis=0)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image)
        loss = predictions[:, class_idx]

    # Compute the gradients of the target class
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply feature maps by pooled gradients
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    # Normalize the heatmap
    heatmap = np.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap)
    return heatmap.numpy()

# Visualize Grad-CAM
def display_gradcam(image, heatmap):
    plt.figure(figsize=(10, 5))

    # Original Image
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis("off")

    # Overlay Grad-CAM
    plt.subplot(1, 2, 2)
    plt.imshow(image)
    plt.imshow(heatmap, cmap='jet', alpha=0.5)  # Overlay heatmap
    plt.title("Grad-CAM")
    plt.axis("off")

    plt.show()

# Example usage
image = test_data[0]  # Use one test image
class_idx = np.argmax(test_labels[0])  # Class index of the true label or prediction
heatmap = compute_gradcam(model, image, class_idx)
display_gradcam(image, heatmap)


In [ ]:
print(f"Shape of test_data: {test_data.shape}")


In [ ]:
sample_image = test_data[0]  # Extract a single image
print(f"Shape of sample_image: {sample_image.shape}")


In [ ]:
print(f"Shape of test_data: {test_data.shape}")
print(f"Shape of sample_image: {sample_image.shape}")
print(f"Shape of input_image: {input_image.shape}")


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generates a Grad-CAM heatmap.

    Parameters:
    - img_array: Preprocessed input image as a numpy array.
    - model: Trained model.
    - last_conv_layer_name: Name of the last convolutional layer.
    - pred_index: Index of the class to generate Grad-CAM for. Default is the predicted class.
    """
    last_conv_layer_name = "sequential_3"  # Update with the name of your last conv layer
    grad_model = tf.keras.models.Model(
    [model.inputs],
    [model.get_layer(last_conv_layer_name).output,
    model.get_layer("sequential_3").output])

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Compute gradients
    grads = tape.gradient(class_channel, conv_outputs)

    # Compute channel-wise importance weights
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the output feature map with gradients
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    # Normalize heatmap between 0 and 1
    heatmap = np.maximum(heatmap, 0) / np.max(heatmap)
    return heatmap


def overlay_heatmap(heatmap, original_image, alpha=0.4, colormap='viridis'):
    """
    Overlays the heatmap on the original image.

    Parameters:
    - heatmap: Generated Grad-CAM heatmap.
    - original_image: Original input image (RGB).
    - alpha: Transparency factor for heatmap overlay.
    - colormap: Colormap to use for the heatmap.
    """
    heatmap = np.uint8(255 * heatmap)
    colormap = plt.cm.get_cmap(colormap)
    colormap_heatmap = colormap(heatmap)
    colormap_heatmap = tf.keras.preprocessing.image.array_to_img(colormap_heatmap[..., :3])
    colormap_heatmap = np.array(colormap_heatmap.resize(original_image.shape[0:2]))

    overlay = original_image.copy()
    overlay = tf.keras.utils.img_to_array(overlay)
    overlay = overlay / overlay.max()  # Normalize original image

    # Blend heatmap and original image
    superimposed_image = colormap_heatmap * alpha + overlay * (1 - alpha)
    return np.uint8(255 * superimposed_image / superimposed_image.max())

# Load a sample brain image
img_path = "/content/op.jpg"
original_image = tf.keras.preprocessing.image.load_img(img_path, target_size=IMAGE_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(original_image)
img_array = np.expand_dims(img_array, axis=0) / 255.0  # Normalize

# Generate Grad-CAM heatmap
last_conv_layer_name = "sequential_3"  # Replace with your layer name
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

# Overlay heatmap on the original image
superimposed_img = overlay_heatmap(heatmap, original_image)

# Display results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.title("Original Image")
plt.imshow(original_image)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Grad-CAM")
plt.imshow(superimposed_img)
plt.axis('off')

plt.show()



In [ ]:
print("Conv Outputs Shape:", conv_outputs.shape)
print("Gradients Shape:", grads.shape)


In [ ]:
# Define dummy input with the same shape as your model's input
dummy_input = tf.random.uniform((1, *IMAGE_SIZE, 3))  # Replace IMAGE_SIZE with your model's input size

# Call the model with dummy input
model(dummy_input)

print("Model has been successfully called with a dummy input!")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model

# Function to compute Grad-CAM heatmap
def get_grad_cam_heatmap(model, img_array, layer_name):
    # Create a model that maps the input image to the activations of the target layer and predictions
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )

    # Compute the gradient of the top predicted class for the input image
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        predicted_class = tf.argmax(predictions[0])  # Top predicted class
        loss = predictions[:, predicted_class]

    # Compute the gradient of the loss with respect to the feature map
    grads = tape.gradient(loss, conv_outputs)

    # Pool the gradients over all the axes except the channel axis
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply the pooled gradients by the feature map
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    # Normalize the heatmap between 0 and 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Function to overlay the heatmap on the original image
def display_grad_cam(image, heatmap, alpha=0.4):
    # Rescale the heatmap to 0-255
    heatmap = np.uint8(255 * heatmap)
    heatmap = np.expand_dims(heatmap, axis=-1)
    heatmap = np.repeat(heatmap, 3, axis=-1)  # Convert to 3 channels

    # Resize the heatmap to match the input image size
    heatmap = tf.image.resize(heatmap, (image.shape[1], image.shape[2])).numpy()

    # Overlay the heatmap on the image
    overlay = tf.keras.utils.array_to_img(image[0])  # Convert the image tensor to an image
    overlay = np.array(overlay)
    superimposed_image = heatmap * alpha + overlay

    # Display the original image and the Grad-CAM heatmap
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(overlay / 255.0)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM Heatmap")
    plt.imshow(superimposed_image / 255.0)
    plt.axis('off')
    plt.show()

# Assuming test_dataset is a tuple of (images, labels)
images, labels = test_data  # Unpack dataset into images and labels

# Select a random index
random_index = np.random.randint(0, len(images))

# Get the random image and its corresponding label
img = images[random_index]
label = labels[random_index]

# Prepare the image for the model (add batch dimension)
img_array = np.expand_dims(img, axis=0)

# Ensure the model is initialized
dummy_input = tf.random.uniform((1, *IMAGE_SIZE, 3))
model(dummy_input)  # Initialize all layers

# Choose the target layer
target_layer = "sequential_3"  # Final convolutional layer

# Generate Grad-CAM heatmap
heatmap = get_grad_cam_heatmap(model, img_array, target_layer)

# Display the Grad-CAM heatmap
display_grad_cam(img_array, heatmap)



In [ ]:
# Access the "sequential_3" submodel
sequential_3 = model.get_layer("sequential_3")

# Inspect the layers in sequential_3
for layer in sequential_3.layers:
    print(layer.name)

# Assuming "separable_conv2d_7" is the desired layer
last_conv_layer_name = "separable_conv2d_7"

# Updated Grad-CAM function
def generate_gradcam_heatmap(model, img_array, submodel, last_conv_layer_name, pred_index=None):
    # Create a model mapping input to the activations of the last convolutional layer
    grad_model = tf.keras.models.Model(
        [model.inputs], [submodel.get_layer(last_conv_layer_name).output, model.output]
    )

    # Compute gradient of the target class score with respect to the activations
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradients of the output neuron w.r.t the feature map
    grads = tape.gradient(class_channel, conv_outputs)

    # Mean intensity of the gradients over the feature map (global average pooling)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply the feature map by the gradients' pooled intensities
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize heatmap between 0 and 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Generate heatmap using the extracted submodel layer
heatmap = generate_gradcam_heatmap(model, random_image_array, sequential_3, last_conv_layer_name)

# Visualize and overlay the heatmap
heatmap_img = superimpose_heatmap(random_image_array, heatmap)
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(random_image)
plt.title("Original Image")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(heatmap_img)
plt.title("Grad-CAM Heatmap")
plt.axis("off")
plt.show()


https://www.kaggle.com/datasets/yasserhessein/dataset-alzheimer

# IMPORTS

In [ ]:
!rm -r "/content/dataset-alzheimer.zip" #if already exists delete
!rm -r "/content/dataset"

In [ ]:
!pip install tensorflow
!pip install albumentations
!pip install --upgrade tensorflow

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv2D,Dense,Dropout,BatchNormalization,Flatten,MaxPool2D,MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau,ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ResNet50,DenseNet121,EfficientNetB2,VGG19,Xception,VGG16
from tensorflow.keras import layers
from tensorflow.keras import Model
import cv2

# DATA Download

In [ ]:
!rm -r "/content/dataset-alzheimer.zip" #if already exists delete

In [ ]:
!pip install -q kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json     #giving permission
!kaggle datasets download yasserhessein/dataset-alzheimer
!unzip "/content/dataset-alzheimer.zip" -d "/content/dataset/"  # UNZIP
!rm -r "/content/dataset-alzheimer.zip"

# DATA MANAGEMENT

## Changing data

there are two folders so we are just shifting from old to the new dir

# Model

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sn
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Flatten, Dropout, Activation, Conv2D, MaxPooling2D, LeakyReLU
from tensorflow.keras.callbacks import TensorBoard
import pickle
from PIL import Image
import time
import os
from PIL import Image
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Device:', tpu.master())
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
except:
    strategy = tf.distribute.get_strategy()

In [ ]:
IMAGE_SIZE = [176,208]               # input image dimensions                                              (176,208)
BATCH_SIZE = 5000
test_split_percent = .1       # % of total data for testing                                         .1
validation_split_percent = .2 # % of total data for validation                                      .2
zoom = [.99,1.01]             # zoom range (for a fixed zoom put the same value in both parameters)[.99,1.01]
bright_range = [.8,1.2]       # brightness range                                                    [.8,1.2]
layers_unlocked = True        # unlock the imported pre-training layers?                            False
lr = 0.01                   # learning rate for optimizer                                         0.0001
batch = 20                    # batch size for model fitting                                        20
EPOCHS = 50                     # number of epochs to run                                             50

## train & test

In [ ]:
train_dr = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255,fill_mode='constant',cval=0,
                                                           brightness_range=bright_range,zoom_range=zoom,
                                                           data_format='channels_last',zca_whitening=False)

train_data_gen = train_dr.flow_from_directory(directory="/content/dataset/Alzheimer_s Dataset/train/",target_size=IMAGE_SIZE,batch_size=BATCH_SIZE)

# Change to zoom = [1,1] to use normal test data
test_dr = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255,fill_mode='constant',cval=0,zoom_range=[1,1],
                                                          data_format='channels_last')
test_data_gen = test_dr.flow_from_directory(directory="/content/dataset/Alzheimer_s Dataset/test",target_size=IMAGE_SIZE,batch_size=BATCH_SIZE,
                                           shuffle = False) # test data should not be shuffle

In [ ]:
train_data,train_labels =  train_data_gen.__next__()
test_data,test_labels = test_data_gen.__next__()

In [ ]:
total_data = np.concatenate((train_data,test_data))
total_labels = np.concatenate((train_labels,test_labels))
print(total_data.shape)
print(total_labels.shape)

In [ ]:
initial_split = test_split_percent+validation_split_percent
test_val_split = test_split_percent/initial_split

# split into training and (test + validation)
train_data, test_val_data, train_labels, test_val_labels = train_test_split(total_data,total_labels,
                                                                            test_size=initial_split)

# split (test + validation) into test and validation sets
test_data, val_data, test_labels, val_labels = train_test_split(test_val_data,test_val_labels,
                                                                test_size=test_val_split)

print('train: ',train_data.shape)
print('validation',val_data.shape)
print('test',test_data.shape)

In [ ]:
print(train_data.shape)
print(train_labels.shape)
print(val_data.shape)
print(val_labels.shape)
print(test_data.shape)
print(test_labels.shape)

In [ ]:
plt.subplot(221)
plt.imshow(train_data[1,:,:,:])
plt.subplot(222)
plt.imshow(train_data[2,:,:,:])
plt.subplot(223)
plt.imshow(val_data[3,:,:,:])
plt.subplot(224)
plt.imshow(val_data[4,:,:,:])
plt.show()
plt.subplot(221)
plt.imshow(test_data[5,:,:,:])
plt.subplot(222)
plt.imshow(test_data[154,:,:,:])

In [ ]:
print(np.amax(train_data))
print(np.amin(train_data))
print(np.amax(val_data))
print(np.amin(val_data))

In [ ]:
plt.subplot(141)
plt.imshow(train_data[3,:,:,0])
plt.subplot(142)
plt.imshow(train_data[3,:,:,1])
plt.subplot(143)
plt.imshow(train_data[3,:,:,2])
plt.subplot(144)
plt.imshow(train_data[3,:,:,:])

## filters

In [ ]:
def conv_block(filters):
    block = tf.keras.Sequential([
        tf.keras.layers.SeparableConv2D(filters, 3, activation='relu', padding='same'),
        tf.keras.layers.SeparableConv2D(filters, 3, activation='relu', padding='same'),
        #tf.keras.layers.SeparableConv2D(filters, 3, activation='relu', strides=2, padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D()
    ])

    return block

In [ ]:
def dense_block(units, dropout_rate):
    block = tf.keras.Sequential([
        tf.keras.layers.Dense(units, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(dropout_rate)
    ])

    return block

## model 2

In [ ]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(*IMAGE_SIZE, 3)),

        tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPool2D(),

        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPool2D(),

        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPool2D(),

        tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPool2D(),

        # tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same'),
        # tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same'),
        # tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same'),
        # tf.keras.layers.MaxPool2D(),


        tf.keras.layers.Flatten(),
        dense_block(512, 0.7),
        dense_block(128, 0.5),
        dense_block(64, 0.3),

        tf.keras.layers.Dense(4, activation='softmax')
    ])

    return model

In [ ]:
with strategy.scope():
    model = build_model()

    METRICS = [tf.keras.metrics.BinaryAccuracy(name='accuracy'),tf.keras.metrics.Precision(name='precision'),
               tf.keras.metrics.Recall(name='recall'),tf.keras.metrics.AUC(name='auc')]


    model.compile(
        optimizer='adam',
        loss=tf.losses.CategoricalCrossentropy(),
        metrics=METRICS
    )


In [ ]:
def exponential_decay(lr0, s):
    def exponential_decay_fn(epoch):
        return lr0 * 0.1 **(epoch / s)
    return exponential_decay_fn

exponential_decay_fn = exponential_decay(0.01, 20)

lr_scheduler = tf.keras.callbacks.LearningRateScheduler(exponential_decay_fn)

# checkpoint_cb = tf.keras.callbacks.ModelCheckpoint("/content/alzheimer_model.keras", save_best_only=True)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "/content/alzheimer_model.keras",  # Path to save the model
    save_best_only=True,               # Save only the best model (based on a metric)
    save_weights_only=False,           # Save both architecture + weights (set to True for only weights)
    monitor='val_loss',                # Monitor validation loss or any other metric
    mode='min',                        # Mode for monitoring (e.g., 'min' to minimize val_loss)
    verbose=1                          # Show progress in the output
)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

### train

In [ ]:
model_history = model.fit(train_data,train_labels,validation_data=(val_data,val_labels),
                             epochs=EPOCHS,batch_size=batch, shuffle=True)